In [ ]:
# imports
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVR, SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    KFold, GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)

In [ ]:
#import and basic information about the dataset
DATA_PATH = "../data/merged_dataset.parquet"

df = pl.read_parquet(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head()

In [ ]:
# Creating a full grid of all hexes and time buckets
SPATIAL_COL = "h3_hex_id"
TIME_BUCKET_COL = "bin_1h"

all_hexes = df[SPATIAL_COL].drop_nulls().unique()
all_buckets = pl.datetime_range(
    df[TIME_BUCKET_COL].min(),
    df[TIME_BUCKET_COL].max(),
    interval="1h",
    eager=True,
)
# filling the grid with observed trip counts
full_grid = pl.DataFrame({SPATIAL_COL: all_hexes}).join(
    pl.DataFrame({TIME_BUCKET_COL: all_buckets}), how="cross"
)

observed = (
    df.group_by([SPATIAL_COL, TIME_BUCKET_COL])
    .agg(pl.len().alias("trip_count"))
)
# filling the NaN values with 0
demand = full_grid.join(observed, on=[SPATIAL_COL, TIME_BUCKET_COL], how="left")
demand = demand.with_columns(pl.col("trip_count").fill_null(0).cast(pl.Int64))

n_total = demand.height
n_zero = demand.filter(pl.col("trip_count") == 0).height

print(f"Gitteranzahl: {n_total:,}")

print(f"Volles Gitter (Hex x 1h-Bucket): {n_total:,}")
print(f"Davon mit 0 Trips: {n_zero:,} ({n_zero / n_total * 100:.1f}%)")
print(f"\nNachfrage-Verteilung:")
print(demand.select("trip_count").describe())

In [ ]:
# target is the raw trip_count (regression) - no class binning needed
tc = demand["trip_count"]
print(f"trip_count als direktes Regressions-Target:")
print(f"  Bereich:   {tc.min()} - {tc.max()}")
print(f"  Mittelwert: {tc.mean():.2f}")
print(f"  Median:     {tc.median():.2f}")
print(f"  Std:        {tc.std():.2f}")

# visualize the distribution (many zeros / right-skewed)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(demand["trip_count"].to_numpy(), bins=50, color="#2980b9")
axes[0].set_title("trip_count Verteilung")
axes[0].set_xlabel("trip_count")
axes[0].set_ylabel("Anzahl")

non_zero = demand.filter(pl.col("trip_count") > 0)["trip_count"].to_numpy()
axes[1].hist(non_zero, bins=50, color="#27ae60")
axes[1].set_title("trip_count Verteilung (nur > 0)")
axes[1].set_xlabel("trip_count")
axes[1].set_ylabel("Anzahl")
plt.tight_layout()
plt.show()

In [ ]:
# adding time and spatial features
df = df.with_columns(pl.col(TIME_BUCKET_COL).cast(pl.Datetime))

WEATHER_COLS = ["temperature_2m", "precipitation"]
HOLIDAY_COL = "is_holiday"
SPATIAL_FEATURE_COLS = []
CYCLIC_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]
WEEKEND_COL = "is_weekend"

# aggregation expressions (placeholder - add pl.col("temperature_2m").mean(), etc.)
agg_exprs = []


print(f"Aggregeierte Features: {agg_exprs}")

if agg_exprs:
    features = df.group_by([SPATIAL_COL, TIME_BUCKET_COL]).agg(agg_exprs)
else:
    features = df.select([SPATIAL_COL, TIME_BUCKET_COL]).unique()

model_df = demand.join(features, on=[SPATIAL_COL, TIME_BUCKET_COL], how="left")
print(f"Model-Dataset Shape: {model_df.shape}")
print(f"NaN pro Spalte:\n{model_df.null_count()}")
model_df.head()

In [ ]:
print(f"Einzigartige {SPATIAL_COL}-Werte: {model_df[SPATIAL_COL].n_unique():,}")
print(f"Feature-Spalten für das Modell: {list(agg_exprs)}")
print(f"\nHinweis: Target-Encoding für {SPATIAL_COL} wird nach dem Train/Test-Split berechnet.")
print("Die Spalte 'hex_target_mean' wird dort hinzugefügt.")

In [ ]:
# final feature columns for modeling
FEATURE_COLS = ["temperature_2m", "precipitation", "is_holiday", "is_weekend", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]

print(f"Finale Feature-Spalten ({len(FEATURE_COLS)}): {FEATURE_COLS}")

# target is directly the trip_count (no LabelEncoder, no classes)
model_df = model_df.with_columns(pl.col("trip_count").cast(pl.Float64).alias("target"))

model_df_clean = model_df.drop_nulls(subset=FEATURE_COLS + ["target"])


In [ ]:
# sorting by time to ensure proper train/test split
model_df_clean = model_df_clean.sort(TIME_BUCKET_COL)

split_idx = int(model_df_clean.height * 0.8)

train_df = model_df_clean[:split_idx]
test_df = model_df_clean[split_idx:]
# recalculating mean to capture only train
hex_mean = (
    train_df.group_by(SPATIAL_COL)
    .agg(pl.col("trip_count").mean().alias("hex_target_mean"))
)
global_mean = train_df["trip_count"].mean()

train_df = train_df.join(hex_mean, on=SPATIAL_COL, how="left").with_columns(
    pl.col("hex_target_mean").fill_null(global_mean)
)
test_df = test_df.join(hex_mean, on=SPATIAL_COL, how="left").with_columns(
    pl.col("hex_target_mean").fill_null(global_mean)
)

FEATURE_COLS_FINAL = FEATURE_COLS + ["hex_target_mean"]
print(f"Finale Features (mit Target-Encoding): {FEATURE_COLS_FINAL}")

X_train_full = train_df.select(FEATURE_COLS_FINAL).to_numpy()
y_train_full = train_df["target"].to_numpy()
X_test = test_df.select(FEATURE_COLS_FINAL).to_numpy()
y_test = test_df["target"].to_numpy()

print(f"\nTrain: {X_train_full.shape[0]:,} Zeilen")
print(f"Test:  {X_test.shape[0]:,} Zeilen")
print(f"\nTrain Target-Statistik:")
print(f"  Mittelwert: {y_train_full.mean():.2f}  Median: {np.median(y_train_full):.1f}  Std: {y_train_full.std():.2f}")
print(f"Test Target-Statistik:")
print(f"  Mittelwert: {y_test.mean():.2f}  Median: {np.median(y_test):.1f}  Std: {y_test.std():.2f}")

In [ ]:
# grid search with 100000 samples
GRID_SAMPLE_SIZE = 100000
np.random.seed(7)
if len(X_train_full) > GRID_SAMPLE_SIZE:
    sample_idx = np.random.choice(len(X_train_full), size=GRID_SAMPLE_SIZE, replace=False)
    X_train_sub = X_train_full[sample_idx]
    y_train_sub = y_train_full[sample_idx]
else:
    X_train_sub = X_train_full
    y_train_sub = y_train_full

print(f"Grid Search Sample: {X_train_sub.shape[0]:,} Zeilen")

In [ ]:
pipe_linear = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", LinearSVR(dual="auto", max_iter=10000, random_state=7)),
])

param_grid_linear = {
    "svr__C": [0.01, 0.1, 1, 10],
    "svr__epsilon": [0.1, 1, 10],
}

cv = KFold(n_splits=5, shuffle=True, random_state=7)

grid_linear = GridSearchCV(
    pipe_linear,
    param_grid_linear,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

print("=== LinearSVR Baseline ===")
grid_linear.fit(X_train_sub, y_train_sub)
print(f"Best Params: {grid_linear.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_linear.best_score_):.4f}")

In [ ]:
pipe_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", max_iter=-1)),
])

param_grid_rbf = {
    "svr__C": [0.1, 1, 10],
    "svr__gamma": ["scale", 0.01, 0.1, 1],
    "svr__epsilon": [0.1, 1, 10],
}

grid_rbf = GridSearchCV(
    pipe_rbf,
    param_grid_rbf,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

print(" RBF Kernel")
grid_rbf.fit(X_train_sub, y_train_sub)
print(f"Best Params: {grid_rbf.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_rbf.best_score_):.4f}")

In [ ]:
pipe_poly = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="poly", max_iter=-1)),
])

param_grid_poly = {
    "svr__C": [0.1, 1, 10],
    "svr__degree": [2, 3],
    "svr__coef0": [0, 1],
    "svr__epsilon": [0.1, 1, 10],
}

grid_poly = GridSearchCV(
    pipe_poly,
    param_grid_poly,
    cv=cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

print("Polynomial Kernel")
grid_poly.fit(X_train_sub, y_train_sub)
print(f"Best Params: {grid_poly.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_poly.best_score_):.4f}")

In [ ]:
results = pl.DataFrame({
    "Model": ["LinearSVR", "RBF", "Polynomial"],
    "Best CV RMSE": [
        np.sqrt(-grid_linear.best_score_),
        np.sqrt(-grid_rbf.best_score_),
        np.sqrt(-grid_poly.best_score_),
    ],
    "Best Params": [
        grid_linear.best_params_,
        grid_rbf.best_params_,
        grid_poly.best_params_,
    ],
})
display(results)

# lower RMSE = better
best_idx = results["Best CV RMSE"].arg_min()
best_name = results["Model"][best_idx]
print(f"\nBestes Modell: {best_name} (CV RMSE: {results['Best CV RMSE'][best_idx]:.4f})")

In [ ]:
grids = {"LinearSVR": grid_linear, "RBF": grid_rbf, "Polynomial": grid_poly}
best_grid = grids[best_name]

best_model = best_grid.best_estimator_
best_model.fit(X_train_full, y_train_full)

print(f"Finales Modell ({best_name}) auf {X_train_full.shape[0]:,} Trainingszeilen gefittet.")

In [ ]:
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2:   {r2:.4f}")

# MAPE only on rows with actual > 0 (avoid division by zero)
mask_nonzero = y_test > 0
if mask_nonzero.sum() > 0:
    mape = np.mean(np.abs((y_test[mask_nonzero] - y_pred[mask_nonzero]) / y_test[mask_nonzero])) * 100
    print(f"Test MAPE (nur trip_count > 0): {mape:.2f}%")

In [ ]:
# Predicted-vs-Actual and Residual Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicted vs Actual
ax = axes[0]
ax.scatter(y_test, y_pred, s=8, alpha=0.3, color="#2980b9")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", lw=1.5, label="perfekte Vorhersage")
ax.set_xlabel("Tatsaechlicher trip_count")
ax.set_ylabel("Vorhergesagter trip_count")
ax.set_title(f"Predicted vs Actual - {best_name}")
ax.legend()

# Residuals
ax = axes[1]
residuals = y_test - y_pred
ax.scatter(y_pred, residuals, s=8, alpha=0.3, color="#e74c3c")
ax.axhline(0, color="black", lw=1, linestyle="--")
ax.set_xlabel("Vorhergesagter trip_count")
ax.set_ylabel("Residuum (tatsaechlich - vorhergesagt)")
ax.set_title(f"Residual Plot - {best_name}")

plt.tight_layout()
plt.show()